In [98]:
import torch 
from torch import nn
from torch.nn import functional as F

In [119]:
PATH = 'model.pth'
#Hyperparameters
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
dropout = 0.2
n_heads = 6
n_layer = 6

In [100]:
with open('input.txt', 'r', encoding='utf-8') as file:
    text = file.read()

In [101]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [102]:
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
def encode (s):
    return [stoi[c] for c in s]
def decode (l):
    return ''.join([itos[i] for i in l])
print(encode("hello world"))
print(decode(encode("hello world")))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [103]:
data = torch.tensor(encode(text), dtype=torch.long, device=device)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [104]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
train_data.shape, val_data.shape

(torch.Size([1003854]), torch.Size([111540]))

In [105]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [106]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k= self.key(x)   #(B,T,head_size)
        q= self.query(x) #(B,T,head_size)
        v= self.value(x) #(B,T,head_size)
        wei = q @ k.transpose(-2,-1)* n_embd**-0.5 #(B,T,head_size) @ (B, head_size,T) -> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v #(B,T,T) @ (B,T,C) -> (B,T,C)
        return out

In [107]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [108]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

In [109]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class ShakespeareLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(n_layer)])
        self.ln_final = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape
        tok_emb = self.token_embedding_table(idx) #(B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  #(T,C)
        x = tok_emb + pos_emb #(B,T,C)
        x = self.blocks(x) #(B,T,C)
        x = self.ln_final(x) #(B,T,C)
        logits = self.lm_head(x) #(B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
m = ShakespearLanguageModel(vocab_size=len(chars)).to(device)

In [112]:
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits, loss = m(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

In [113]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [114]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long, device=device), max_new_tokens=500)[0].tolist()))


XRVQwWMBq:NYx:IXJGj$f'-b'EKFLyOD:aLmjoeZfcqeE,MDN;uDSMG3AiYB;MxRbCDbfZLVYo.Noc VPXvilaRPD-TK  hmT'UReKu.zEfOXQ
zg$Ano.pvH
WMXBScprhQ:&$rl$Ohb3weJYcI
N?DnHw!EkgVp ?iGobiW3dg;LhuDiFnuMMfAiLrTDZjMdwc?MXgxJ:Djr-ikTDaFi$lNOE!&TUOOZe$neXnogJYfC,;cLhwstkJ&&mv?PJ:YwVphhfba:zbEzY?-yEiJrYOLjQxviYO3?N'tQBppVhkOAUy$cbw:BpN-rEkHFLn&AswuZz'YApF,Oc3HAfYleJyM?OhfQc;mKHGLYAVYtg:aAxzeP$NGct-UMtw3y?sOEz3!drXYYEV:QNPfc:Ru:olGP.mqvcO?kLV
kJYNmrK-YKpF
Wtxci',-sy,SpgxwJ-ckjNMLz'ApnXNGWnoGyQFYGg :$:$:eOE'noNY!WROWYQIu.


In [118]:
def trainloop():
    for steps in range(max_iters):
        xb, yb = get_batch('train')

        logits, loss = m(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if steps % eval_interval == 0 or steps == max_iters - 1:
            losses = estimate_loss()
            print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

In [121]:
m.load_state_dict(torch.load(PATH))
# trainloop()
# torch.save(m.state_dict(), PATH)

<All keys matched successfully>

In [122]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long, device=device), max_new_tokens=500)[0].tolist()))



COMINIUS:
Why,
This affections are coming by you.

MENENIUS:
In brief, the most patient of the air
Which not ever wrom the winness of him; I will.
Cold famillow guod to you, friar: not advice
To warn it glass your grace. Would you do me it so?

CORIOLANUS:
No, so.

AUTOLYCUS:
I do burn it, looking,
The stocks of fire and unaking scutny,
How are they bound?

CORIOLANUS:
Your bjects;
This diade is very like. Now, sir, it shall die.

A Players:
Resolved; and you have done to stage him hence;
For t
